# Data Validation
**This file will validate the data exploration efforts to ensure accurate data handling.**

#### Task 12: independently comparing the record counts from the raw dataset to the prepared dataset, taking into consideration the removal of duplicated rows.

Utilizing 01_data_explortation prepared dataset.

In [23]:
import pandas as pd

#loading prepared dataset
df_prepared = pd.read_csv("data/restaurants_prepared.csv",
                          dtype={"ZIPCODE": str, "Community Board": str, "Council District": str,
                                 "Census Tract": str, "BIN": str, "BBL": str},
                          parse_dates=["INSPECTION DATE", "GRADE DATE", "RECORD DATE"],
                          low_memory=False)

# loading raw dataset
df_raw = pd.read_csv("data/restaurant_inspections.csv")

#loading excluded records
df_exclusion_log = pd.read_csv("handoff/duplicate_exclusion_log.csv")

Validating the record counts from _01_data_exploration_.

In [24]:
print("Raw records:", len(df_raw))
print("Prepared records:", len(df_prepared))

print("Expected prepared records:",
      len(df_raw) - len(df_exclusion_log))

Raw records: 296012
Prepared records: 295843
Expected prepared records: 295843


Checking source identifiers (number of source IDs should equal number of records in prepared dataset).

In [25]:
print("Unique source IDs:", df_prepared["SOURCE_ROW_ID"].nunique())

print("Missing source IDs:", df_prepared["SOURCE_ROW_ID"].isna().sum())

Unique source IDs: 295843
Missing source IDs: 0


#### Task 14: Validating Categorical Values

This code looks at the categorial variables for missing values and data inconsistencies.

Task 13 is located in _03_data_dictionary_.

In [38]:
#prelimiary check for unique values
categorical_col = ["BORO", "CUISINE DESCRIPTION", "ACTION", "INSPECTION TYPE",
                   "GRADE", "CRITICAL FLAG", "VIOLATION CODE", "VIOLATION DESCRIPTION"]

for col in categorical_col:
    print("\n")
    print(df_prepared[col].value_counts(dropna=False))



BORO
Manhattan        109608
Brooklyn          74873
Queens            74192
Bronx             27226
Staten Island      9593
0                   351
Name: count, dtype: int64


CUISINE DESCRIPTION
American              44081
Chinese               29107
Coffee/Tea            21582
Pizza                 17033
Latin American        14854
                      ...  
Nuts/Confectionary       32
Czech                    19
Polynesian               10
Haute Cuisine             8
Chimichurri               4
Name: count, Length: 91, dtype: int64


ACTION
Violations were cited in the following area(s).                                                                                       275767
Establishment Closed by DOHMH. Violations were cited in the following area(s) and those requiring immediate action were addressed.     10839
NaN                                                                                                                                     3714
No violations were reco

Preliminary checks show some missing values, but the text data appears consistent. Each column will be checked for amount of nulls.

In [39]:
#check for null values
categorical_col = ["BORO", "CUISINE DESCRIPTION", "ACTION", "INSPECTION TYPE",
                   "GRADE", "CRITICAL FLAG", "VIOLATION CODE", "VIOLATION DESCRIPTION"]

for col in categorical_col:
    print(f"\nAmount of {col} nulls:")
    print(df_prepared[col].isna().sum())


Amount of BORO nulls:
0

Amount of CUISINE DESCRIPTION nulls:
3794

Amount of ACTION nulls:
3714

Amount of INSPECTION TYPE nulls:
3714

Amount of GRADE nulls:
150537

Amount of CRITICAL FLAG nulls:
0

Amount of VIOLATION CODE nulls:
6444

Amount of VIOLATION DESCRIPTION nulls:
6444


#### Task 15: Investigate missing-value patterns

This code aims to understand the relationships between columns with missing values.

Will first separate restaurants based on the already created _AWAITING_INSPECTION_ flag, and analyze the relationships between the restaurants inspection status and their action, inspection type, and grade statuses.

In [41]:
#df for restaurants awaiting inspection
df_awaiting = df_prepared[df_prepared["AWAITING_INSPECTION"] == True]

#df for restaurants that have already received an inspection
df_inspected = df_prepared[df_prepared["AWAITING_INSPECTION"] == False]

inspected_cols = [
    "ACTION",
    "INSPECTION TYPE",
    "GRADE",
    "VIOLATION CODE",
    "VIOLATION DESCRIPTION",
    "CUISINE DESCRIPTION"
]

print(df_inspected[inspected_cols].isna().sum())

ACTION                        0
INSPECTION TYPE               0
GRADE                    146823
VIOLATION CODE             2730
VIOLATION DESCRIPTION      2730
CUISINE DESCRIPTION          80
dtype: int64


Now charting relationship between all categorical variables with nulls and their inspection status.

In [42]:
df_prepared.groupby("AWAITING_INSPECTION")[["ACTION", "INSPECTION TYPE", "CUISINE DESCRIPTION", 
                                            "GRADE", "VIOLATION CODE", "VIOLATION DESCRIPTION"]].apply(
    lambda x: x.isna().sum()
)

,ACTION,INSPECTION TYPE,CUISINE DESCRIPTION,GRADE,VIOLATION CODE,VIOLATION DESCRIPTION
AWAITING_INSPECTION,,,,,,
False,0,0,80,146823,2730,2730
True,3714,3714,3714,3714,3714,3714


Findings:
- Restaurants that lack _ACTION_, _INSPECTION TYPE_, _CUISINE DESCRIPTION_, __AND__ _GRADE_ values are those that are awaiting an inspection (or have the placeholder 1900-01-01 as their inspection date).
- Of restaurants that have been inspected, 80 lack a _CUISINE DESCRIPTION_, AND 146,823 lack a GRADE.

How many inspected resturants lacking a score also lack a grade?

In [35]:
pd.crosstab(df_inspected["SCORE"].isna(),
            df_inspected["GRADE"].isna(),
            rownames = ["Score Missing"],
            colnames = ["Grade Missing"],
            margins = True)

Grade Missing,False,True,All
Score Missing,,,
False,145295,133328,278623
True,11,13495,13506
All,145306,146823,292129


Findings:
- Restaurants missing __both__ grades and scores total 13,495.
- Restaurants missing __only__ grades total 133,328.
- Restaurants missing __only__ scores total 11.

How many inspected restaurants lacking a violation code also lack a violation description?

In [43]:
pd.crosstab(df_inspected["VIOLATION CODE"].isna(),
            df_inspected["VIOLATION DESCRIPTION"].isna(),
            rownames = ["Code Missing"],
            colnames = ["Description Missing"],
            margins = True)

Description Missing,False,True,All
Code Missing,,,
False,289399,0,289399
True,0,2730,2730
All,289399,2730,292129


Findings:
- Restaurants missing a violation description are __also__ missing a code, there are 2730 inspected restaurants with no violation details at all (code and description).

How many inspected restaurants are missing violation details and a score?

In [44]:
pd.crosstab(df_inspected["SCORE"].isna(),
            df_inspected["VIOLATION CODE"].isna(),
            rownames = ["Score Missing"],
            colnames = ["Code Missing"],
            margins = True)

Code Missing,False,True,All
Score Missing,,,
False,276811,1812,278623
True,12588,918,13506
All,289399,2730,292129


Findings:
- Restaurants missing __only__ a violation code (and thus description) total 1,812.
- Restaurants missing __only__ a score total 12,588.
- Restaurants missing __both__ a score and violation code total 918.

#### Task 16: Missing values handled for each column

Of the categorical columns examined, the following contain null values.
- ACTION
- INSPECTION TYPE
- GRADE
- VIOLATION CODE
- VIOLATION DESCRIPTION
- CUISINE DESCRIPTION

Missing cuisine descriptions will be filled with an "Unknown" placeholder. The cuisine descriptions currently do not appear to answer the main question of:
- _What factors are associated with restaurant food safety performance in NYC, and can historical inspection records help identify restaurants at greater risk of poor inspection outcomes?_

In [19]:
df_prepared["CUISINE DESCRIPTION"] = (df_prepared["CUISINE DESCRIPTION"].fillna("Unknown"))